# Run all model build/train/eval steps

**Instructions:** Select the Python 3.13 kernel (the environment where TensorFlow is installed) and run every cell in order. This notebook will attempt to build baseline profiles, train the attack-type classifier, train the LSTM (if TensorFlow is available), run evaluation, and list produced artifacts.

In [ ]:
import sys, os
# Ensure repo root is on sys.path and use it as the working directory
repo_root = os.path.abspath('..')
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
os.chdir(repo_root)

print('Repo root:', repo_root)
print('Working directory:', os.getcwd())

Repo root: d:\Anomaly-Detection


In [41]:
import sys
print('Python:', sys.version.replace('\n', ' '))
try:
    import tensorflow as tf
    print('TensorFlow available:', tf.__version__)
    tf_available = True
except Exception as e:
    print('TensorFlow not available in this kernel:', e)
    tf_available = False

Python: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
TensorFlow available: 2.21.0


In [ ]:
# Helper that attempts to import and call a named function, else runs the module with -m
import subprocess, sys, os
from importlib import import_module

def run_module_or_call(module_name, func_name=None, call_with_df=None):
    try:
        mod = import_module(module_name)
        if func_name and hasattr(mod, func_name):
            print(f'Calling {module_name}.{func_name}()')
            getattr(mod, func_name)()
            return True
        # if asked to call with df, try common function names
        if call_with_df is not None:
            for cand in ['build_baseline_profiles', 'train', 'train_model', 'train_lstm_model', 'main']:
                if hasattr(mod, cand):
                    fn = getattr(mod, cand)
                    try:
                        print(f'Calling {module_name}.{cand}(df)')
                        fn(call_with_df)
                        return True
                    except TypeError:
                        # signature mismatch, continue
                        pass
        print(f'Imported {module_name} but did not call a function directly')
    except Exception as e:
        print(f'Import failed for {module_name}:', e)
    # Fallback to running as a module
    try:
        print(f'Running as module: {module_name}')
        subprocess.run([sys.executable, '-m', module_name], check=True)
        return True
    except Exception as e:
        print(f'Running module {module_name} failed:', e)
        return False

# Load a dataset for baseline building (prefer processed; fallback to raw labeled CSV)
import pandas as pd
data_path = None
if os.path.exists('data/processed/processed_logs.csv'):
    data_path = 'data/processed/processed_logs.csv'
elif os.path.exists('data/raw/cyber_logs.csv'):
    data_path = 'data/raw/cyber_logs.csv'
elif os.path.exists('data/raw/cybersecurity_dataset.csv'):
    data_path = 'data/raw/cybersecurity_dataset.csv'
if data_path is None:
    print('No dataset found at data/processed/processed_logs.csv, data/raw/cyber_logs.csv, or data/raw/cybersecurity_dataset.csv')
    df = None
else:
    print('Loading dataset for baseline:', data_path)
    df = pd.read_csv(data_path)
    print('Loaded rows:', len(df))

# Baseline profiles: prefer the module entrypoint that saves the artifact
print('--- Building baseline profiles ---')
ok = False
try:
    mod = import_module('src.baseline_profiling')
    if hasattr(mod, 'create_baseline_profile_artifact'):
        try:
            print('Calling src.baseline_profiling.create_baseline_profile_artifact()')
            mod.create_baseline_profile_artifact()
            ok = True
        except Exception as e:
            print('Direct artifact creation failed:', e)
            ok = run_module_or_call('src.baseline_profiling', None)
    elif hasattr(mod, 'build_baseline_profiles') and df is not None:
        try:
            print('Calling src.baseline_profiling.build_baseline_profiles(df)')
            mod.build_baseline_profiles(df)
            ok = True
        except Exception as e:
            print('Direct call failed:', e)
            ok = run_module_or_call('src.baseline_profiling', None)
    else:
        ok = run_module_or_call('src.baseline_profiling', None)
except Exception as e:
    print('Baseline build failed:', e)
    ok = False
print('Baseline step ok=', ok)

No dataset found at data/processed/processed_logs.csv or data/raw/cyber_logs.csv; baseline build may fail
--- Building baseline profiles ---
Imported src.baseline_profiling but did not call a function directly
Running as module: src.baseline_profiling
Running module src.baseline_profiling failed: Command '['c:\\Program Files\\Python313\\python.exe', '-m', 'src.baseline_profiling']' returned non-zero exit status 1.
Baseline step ok= False


In [43]:
# Attack-type classifier training
print('--- Training attack-type classifier ---')
ok = run_module_or_call('src.attack_type_classifier', 'train_attack_type_classifier')
# Ensure artifact exists; try import+call if not
import os
if not os.path.exists('trained_models/attack_type_classifier.pkl'):
    try:
        from importlib import import_module
        mod = import_module('src.attack_type_classifier')
        if hasattr(mod, 'train_attack_type_classifier'):
            print('Calling train_attack_type_classifier() directly')
            mod.train_attack_type_classifier()
            ok = os.path.exists('trained_models/attack_type_classifier.pkl')
    except Exception as e:
        print('Direct call failed:', e)
print('Attack-type training ok=', ok)

--- Training attack-type classifier ---
Calling src.attack_type_classifier.train_attack_type_classifier()
Calling train_attack_type_classifier() directly
Attack-type training ok= False


In [44]:
# LSTM training (only if TF available)
print('--- Training LSTM model (if TF available) ---')
if 'tf_available' in globals() and tf_available:
    import importlib
    try:
        mod = importlib.import_module('src.lstm_sequence_model')
        called = False
        for cand in ['train_lstm_model', 'train', 'train_model', 'main']:
            if hasattr(mod, cand):
                try:
                    print(f'Calling src.lstm_sequence_model.{cand}()')
                    getattr(mod, cand)()
                    called = True
                    break
                except TypeError:
                    if 'df' in globals() and df is not None:
                        try:
                            print(f'Calling {cand}(df)')
                            getattr(mod, cand)(df)
                            called = True
                            break
                        except Exception:
                            pass
        if not called:
            print('No callable train function found in module; falling back to -m')
            import subprocess, sys
            try:
                subprocess.run([sys.executable, '-m', 'src.lstm_sequence_model'], check=True)
            except Exception as e:
                print('Running module -m failed:', e)
    except Exception as e:
        print('Importing lstm module failed:', e)
else:
    print('Skipping LSTM training: TensorFlow not available in this kernel')

--- Training LSTM model (if TF available) ---
No callable train function found in module; falling back to -m
Running module -m failed: Command '['c:\\Program Files\\Python313\\python.exe', '-m', 'src.lstm_sequence_model']' returned non-zero exit status 1.


In [45]:
# Evaluation step
print('--- Running evaluation script ---')
import os
if not os.path.exists('trained_models/baseline_profile.pkl'):
    print('baseline_profile.pkl not found; evaluation may fail')
ok = run_module_or_call('src.evaluate_models', None)
print('Evaluation ok=', ok)

--- Running evaluation script ---
baseline_profile.pkl not found; evaluation may fail
Import failed for src.evaluate_models: [Errno 2] No such file or directory: 'trained_models/baseline_profile.pkl'
Running as module: src.evaluate_models
Running module src.evaluate_models failed: Command '['c:\\Program Files\\Python313\\python.exe', '-m', 'src.evaluate_models']' returned non-zero exit status 1.
Evaluation ok= False


In [46]:
# Check for artifacts
import os, json
artifacts = [
    'trained_models/attack_type_classifier.pkl',
    'trained_models/lstm_model.keras',
    'trained_models/evaluation_results.json',
]
print('--- Artifact check ---')
for p in artifacts:
    print(p, '->', os.path.exists(p))
if os.path.exists('trained_models/evaluation_results.json'):
    try:
        data = json.load(open('trained_models/evaluation_results.json'))
        print('evaluation_results contains keys:', list(data.keys()))
    except Exception as e:
        print('Failed reading evaluation_results.json', e)

--- Artifact check ---
trained_models/attack_type_classifier.pkl -> False
trained_models/lstm_model.keras -> False
trained_models/evaluation_results.json -> False


In [47]:
# LSTM training (only if TF available)
print('--- Training LSTM model (if TF available) ---')
if 'tf_available' in globals() and tf_available:
    import importlib
    try:
        mod = importlib.import_module('src.lstm_sequence_model')
        called = False
        for cand in ['train_lstm_model', 'train', 'train_model', 'main']:
            if hasattr(mod, cand):
                try:
                    print(f'Calling src.lstm_sequence_model.{cand}()')
                    getattr(mod, cand)()
                    called = True
                    break
                except TypeError:
                    if 'df' in globals() and df is not None:
                        try:
                            print(f'Calling {cand}(df)')
                            getattr(mod, cand)(df)
                            called = True
                            break
                        except Exception:
                            pass
        if not called:
            print('No callable train function found in module; falling back to -m')
            import subprocess, sys
            try:
                subprocess.run([sys.executable, '-m', 'src.lstm_sequence_model'], check=True)
            except Exception as e:
                print('Running module -m failed:', e)
    except Exception as e:
        print('Importing lstm module failed:', e)
else:
    print('Skipping LSTM training: TensorFlow not available in this kernel')

--- Training LSTM model (if TF available) ---
No callable train function found in module; falling back to -m


Running module -m failed: Command '['c:\\Program Files\\Python313\\python.exe', '-m', 'src.lstm_sequence_model']' returned non-zero exit status 1.


In [48]:
# Evaluation step
print('--- Running evaluation script ---')
import os
if not os.path.exists('trained_models/baseline_profile.pkl'):
    print('baseline_profile.pkl not found; evaluation may fail')
ok = run_module_or_call('src.evaluate_models', None)
print('Evaluation ok=', ok)

--- Running evaluation script ---
baseline_profile.pkl not found; evaluation may fail
Import failed for src.evaluate_models: [Errno 2] No such file or directory: 'trained_models/baseline_profile.pkl'
Running as module: src.evaluate_models


Running module src.evaluate_models failed: Command '['c:\\Program Files\\Python313\\python.exe', '-m', 'src.evaluate_models']' returned non-zero exit status 1.
Evaluation ok= False


In [49]:
# Check for artifacts
import os, json
artifacts = [
    'trained_models/attack_type_classifier.pkl',
    'trained_models/lstm_model.keras',
    'trained_models/evaluation_results.json',
]
print('--- Artifact check ---')
for p in artifacts:
    print(p, '->', os.path.exists(p))
if os.path.exists('trained_models/evaluation_results.json'):
    try:
        data = json.load(open('trained_models/evaluation_results.json'))
        print('evaluation_results contains keys:', list(data.keys()))
    except Exception as e:
        print('Failed reading evaluation_results.json', e)

--- Artifact check ---
trained_models/attack_type_classifier.pkl -> False
trained_models/lstm_model.keras -> False
trained_models/evaluation_results.json -> False
